# Two LLMs play TicTacToe
Demonstrating how to use Agents and tools by having two LLMs play TicTacToe

In [16]:
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from agents import Agent, Runner, trace, function_tool
from typing import List

In [17]:
# define structured outputs
class GameBoard(BaseModel):
    List[List[str]]

# TODO need a winner field
class GameResult(BaseModel):
    is_complete: bool

In [18]:
# create Controller Agent's tools first

# display tool
@function_tool
def display_board(board: List[List[str]]):
    """This function tool will display the game board to the screen"""
    for row in board:
        for elem in row:
            print(elem if elem != "" else "_", end="")
        print()
    return {"status":"complete"}

@function_tool
def is_game_over(board: List[List[str]]):
    num_filled = 0
    for row in board:
        for elem in row:
            if elem == 'X' or elem == 'O':
                num_filled += 1
    
    return {"is_complete": num_filled > 4}


# players (each are their own tools)
player_1_instruction= "You are a TicTacToe Player, your symbol is X; You are given a 3x3 two dimensional list and you will replace an empty string, denoted by two double quotes, with your symbol, and then return the two dimensional list containing your input"
player_2_instruction= "You are a TicTacToe Player, your symbol is O; You are given a 3x3 two dimensional list and you will replace an empty string, denoted by two double quotes, with your symbol, and then return the two dimensional list containing your input"
player_1_tool = Agent(name="Player1: X", instructions=player_1_instruction, model="gpt-4.1-mini", output_type=GameBoard).as_tool(tool_name="player_1_tool", tool_description="TicTacToe Player : X")
player_2_tool = Agent(name="Player2: O", instructions=player_2_instruction, model="gpt-4.1-mini", output_type=GameBoard).as_tool(tool_name="player_2_tool", tool_description="TicTacToe Player : O")


In [19]:
# create the Controller Agent
#instruction = """
#You are a TicTacToe Controller.
#You will alternate between your player tools to allow both tools to provide their input.
#In between each turn, you will display the TicTacToe GameBoard with the use of another one of your tools.  
#After displaying the game board (after each turn), you will use another tool to check if the game is over.  If the game is over, stop alternating between players.
#"""

tools=[player_1_tool, player_2_tool, display_board, is_game_over]
instruction = """
You are a TicTacToe Controller.
The GameBoard is a 3x3 two dimentional list, you will maintain a single GameBoard.
The initial board shall contain empty strings only.  
You will alternate between your player tools and provide them the latest GameBoard as input and instruct them to place their symbol somewhere on the provided board. Instruct them to return the updated GameBoard containing their input.
IMPORTANT, you must provide the latest Gameboard as input when calling each player, and only one player tool can write to the gameboard at a time.  You must wait for each player tool to return the GameBoard before calling the next player tool.
After each player provides their input, you will display the TicTacToe GameBoard with the use of another one of your tools and wait for it to be displayed before calling another tool.
With the use of another one of your tools, check if the game is over after every single turn before allowing another player to write to the GameBoard; if it is, terminate the game and do not ask for anymore inputs!
"""
controller_agent = Agent(name="TicTacToe_Controller", instructions=instruction, model="gpt-4.1-mini", tools=tools)

In [21]:
load_dotenv()

with trace("TicTacToe-Test"):
    result = await Runner.run(controller_agent, "run a tictactoe game as instructed")
print(result)

CancelledError: 